# IVAN — Matrix Regression (processed `.npz` → target `.npz`)

Актуальная задача:

- **Вход:** `.npz` с `matrix_data` (2 канала), приводим к (2, H, W) и нормируем по каналам в **[0, 1]**
- **Выход модели:** 2D float-матрица размера **(NZ, NX)** в *raw*-домейне (таргет тоже raw)
- **Таргет:** `.npz` (2D matrix), **без нормализации**
- **Модель:** patch-embedding (stride-conv) + глубокая conv-обработка на patch-grid + U-Net (down/bottleneck/up) + проекция `A*1000 + B*100 + C*10 + D`
- **Визуализация:** 2-channel input + target/pred + A/B/C/D maps

In [ ]:
# 0) Imports
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from iternet.config import DataConfig, GridConfig, ModelConfig, TrainConfig
from iternet.data_discovery import discover_train_test
from iternet.dataset import IternetDataset, collate_batch, collate_single
from iternet.model import IternetUNet
from iternet.pipeline import open_training_data, preprocess_data, init_model, predict_mask
from iternet.train import train_segmentation
from iternet.viz import plot_prediction, plot_target_vs_prediction, plot_two_channel_image

In [ ]:
# 1) Paths and global configs
DATA_DIR = Path(r"C:\Max\Proga\IVAN\data\processed")

# Target matrix size (Z, X)
NZ, NX = 75, 235

GRID = GridConfig(
    look_nx=NX,
    look_nz=NZ,
    x_min=-300.0,
    x_max=300.0,
    z_min=0.0,
    z_max=300.0,
)

# Swin-patch-UNet: patch embedding + window attention + iterative upscaling
MODEL_CFG = ModelConfig(
    in_channels=2,
    patch_size=2,
    base_channels=32,
    depth=5,
    blocks_per_stage=5,
    stem_blocks=3,
    num_heads=4,
    window_size=4,
    output_upsample_stages=3,
    out_channels=1,
)

# Loss weights (regression in compressed output domain, metrics in raw domain)
TRAIN_CFG = TrainConfig(
    epochs=15,
    batch_size=16,
    lr=5e-4,
    weight_decay=1e-4,
    scheduler_name="cosine",
    warmup_epochs=2,
    min_lr_ratio=0.1,
    mse_weight=1.0,
    mae_weight=0.1,
    boundary_weight_factor=1.0,
    boundary_weight_radius=3,
    boundary_loss_weight=0.01,
    device="cuda" if torch.cuda.is_available() else "cpu",
    log_dir=Path("iternet/runs"),
)

print("DATA_DIR:", DATA_DIR)
print("DEVICE:", TRAIN_CFG.device)
print("GRID (Z,X):", GRID.look_nz, GRID.look_nx)

In [ ]:
# 5) Init model (Swin patch-UNet + digit polynomial projection)
# Вход: (B, 2, 29, 47)
# Выход: raw (B, 1, NZ, NX)
# Внутри модель предсказывает:
#   - A/B/C как logits по значениям -9..9
#   - D как свободный residual
# и проецирует в: A*1000 + B*100 + C*10 + D
model = IternetUNet(
    in_channels=MODEL_CFG.in_channels,
    patch_size=MODEL_CFG.patch_size,
    base_channels=MODEL_CFG.base_channels,
    depth=MODEL_CFG.depth,
    blocks_per_stage=MODEL_CFG.blocks_per_stage,
    stem_blocks=MODEL_CFG.stem_blocks,
    num_heads=MODEL_CFG.num_heads,
    window_size=MODEL_CFG.window_size,
    output_upsample_stages=MODEL_CFG.output_upsample_stages,
    out_channels=MODEL_CFG.out_channels,
)

# total / trainable params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total_params)
print("Trainable params:", trainable_params)

In [ ]:
# 2) Discover train/test pairs (.npz input + .npz target)
train_pairs, test_pairs = discover_train_test(DATA_DIR)
print(f"Train pairs: {len(train_pairs)}")
print(f"Test pairs : {len(test_pairs)}")

train_pairs[:3] if train_pairs else None

In [ ]:
# # 3) Sanity-check: load one sample via pipeline and visualize target vs (untrained) prediction
# # Важно: тут НЕ проверяем чекпоинт, потому что его может не быть.

# if not train_pairs:
#     raise RuntimeError("No train pairs found. Check data/processed layout.")

# one = train_pairs[0]
# DATA_ONE = DataConfig(
#     ie2d_res_path=one.ie2d_res,
#     target_matrix_path=one.target_matrix,
#     value_kind="auto",
#     current_a=1.0,
# )
# raw_one = open_training_data(DATA_ONE)
# prep_one = preprocess_data(raw_one, GRID)

# print("meas_values_01:", tuple(prep_one.sample.meas_values_01.shape))
# print("target_norm:", tuple(prep_one.sample.target_matrix_norm.shape))
# print("target_raw:", tuple(prep_one.sample.target_matrix_raw.shape))

# # Untrained model prediction (only to test that shapes work)
# model_untrained = init_model(prep_one, MODEL_CFG)
# pred_untrained = predict_mask(model_untrained, prep_one, device=TRAIN_CFG.device)
# target_raw = prep_one.sample.target_matrix_raw

# fig = plot_target_vs_prediction(
#     target_raw,
#     pred_untrained,
#     title="Sanity check (untrained): target vs prediction",
#     x_coords=prep_one.sample.x_coords,
#     z_coords=prep_one.sample.z_coords,
# )
# plt.show()

In [ ]:
# 4) Build datasets and loaders
# Train loader: shuffle=True
train_ds = IternetDataset(
    samples=train_pairs,
    nx=GRID.look_nx,
    nz=GRID.look_nz,
    grid_overrides={"x_min": GRID.x_min, "x_max": GRID.x_max, "z_min": GRID.z_min, "z_max": GRID.z_max},
)
val_ds = IternetDataset(
    samples=test_pairs,
    nx=GRID.look_nx,
    nz=GRID.look_nz,
    grid_overrides={"x_min": GRID.x_min, "x_max": GRID.x_max, "z_min": GRID.z_min, "z_max": GRID.z_max},
) if test_pairs else None

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_CFG.batch_size,
    shuffle=True,
    collate_fn=collate_batch if TRAIN_CFG.batch_size > 1 else collate_single,
    num_workers=0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_single,
    num_workers=0,
) if val_ds is not None else None

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader) if val_loader is not None else 0)

In [ ]:
# 4.5) Visualize one processed sample (2-channel input)
if len(train_ds) == 0:
    raise RuntimeError("Train dataset is empty")

x0, y0, meta0 = train_ds[100]
print("x0:", tuple(x0.shape), "y0:", tuple(y0.shape), "input_kind:", meta0.get("input_kind"))

fig_in = plot_two_channel_image(x0.numpy(), title=f"Input sample {meta0.get('sample_id')}")
plt.show()

fig_tgt = plot_prediction(meta0.get("target_matrix_raw"), title=f"Target (raw) {meta0.get('sample_id')}", x_coords=meta0.get("x_coords"), z_coords=meta0.get("z_coords"))
plt.show()

In [ ]:
# 6) Train (regression)
# Логи: iternet/runs/<timestamp> (TensorBoard + val_images)
config_dict = {
    "data_dir": str(DATA_DIR),
    "nx": NX,
    "nz": NZ,
    "x_min": GRID.x_min,
    "x_max": GRID.x_max,
    "z_min": GRID.z_min,
    "z_max": GRID.z_max,
    "epochs": TRAIN_CFG.epochs,
    "batch_size": TRAIN_CFG.batch_size,
    "lr": TRAIN_CFG.lr,
    "weight_decay": TRAIN_CFG.weight_decay,
    "scheduler_name": TRAIN_CFG.scheduler_name,
    "warmup_epochs": TRAIN_CFG.warmup_epochs,
    "min_lr_ratio": TRAIN_CFG.min_lr_ratio,
    "mse_weight": TRAIN_CFG.mse_weight,
    "mae_weight": TRAIN_CFG.mae_weight,
    "boundary_weight_factor": TRAIN_CFG.boundary_weight_factor,
    "boundary_weight_radius": TRAIN_CFG.boundary_weight_radius,
    "boundary_loss_weight": TRAIN_CFG.boundary_loss_weight,
    "model_in_channels": MODEL_CFG.in_channels,
    "model_patch_size": MODEL_CFG.patch_size,
    "model_base_channels": MODEL_CFG.base_channels,
    "model_depth": MODEL_CFG.depth,
    "model_blocks_per_stage": MODEL_CFG.blocks_per_stage,
    "model_stem_blocks": MODEL_CFG.stem_blocks,
    "model_num_heads": MODEL_CFG.num_heads,
    "model_window_size": MODEL_CFG.window_size,
    "model_output_upsample_stages": MODEL_CFG.output_upsample_stages,
    "model_out_channels": MODEL_CFG.out_channels,
    "device": TRAIN_CFG.device,
}

history = train_segmentation(
    model=model,
    loader=train_loader,
    epochs=TRAIN_CFG.epochs,
    lr=TRAIN_CFG.lr,
    weight_decay=TRAIN_CFG.weight_decay,
    device=TRAIN_CFG.device,
    log_dir=TRAIN_CFG.log_dir,
    val_loader=val_loader,
    log_every_steps=TRAIN_CFG.log_every_steps,
    mse_weight=TRAIN_CFG.mse_weight,
    mae_weight=TRAIN_CFG.mae_weight,
    boundary_weight_factor=TRAIN_CFG.boundary_weight_factor,
    boundary_weight_radius=TRAIN_CFG.boundary_weight_radius,
    boundary_loss_weight=TRAIN_CFG.boundary_loss_weight,
    scheduler_name=TRAIN_CFG.scheduler_name,
    warmup_epochs=TRAIN_CFG.warmup_epochs,
    min_lr_ratio=TRAIN_CFG.min_lr_ratio,
    config_dict=config_dict,
)

print("Final train loss:", history.losses[-1] if history.losses else None)
print("Final train rmse:", history.rmse[-1] if history.rmse else None)
if history.val_rmse:
    print("Final val rmse:", history.val_rmse[-1])

In [ ]:
# 7) Save checkpoint (after training)
# Формат: {'model': state_dict, 'arch': 'swin_patch_unet_digit_abcd', ...}
CKPT_OUT = Path("iternet/runs") / "model_regression.pt"
CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model": model.state_dict(),
        "arch": "swin_patch_unet_digit_abcd",
        "in_channels": MODEL_CFG.in_channels,
        "patch_size": MODEL_CFG.patch_size,
        "base_channels": MODEL_CFG.base_channels,
        "depth": MODEL_CFG.depth,
        "blocks_per_stage": MODEL_CFG.blocks_per_stage,
        "stem_blocks": MODEL_CFG.stem_blocks,
        "num_heads": MODEL_CFG.num_heads,
        "window_size": MODEL_CFG.window_size,
        "output_upsample_stages": MODEL_CFG.output_upsample_stages,
        "out_channels": MODEL_CFG.out_channels,
    },
    CKPT_OUT,
)

print("Saved:", CKPT_OUT)

In [ ]:
# 8) Test inference on a trained checkpoint (optional)
# ВАЖНО: проверка наличия чекпоинта делается только здесь.

CHECKPOINT_PATH = CKPT_OUT  # можно заменить на путь к любому *.pt

if not CHECKPOINT_PATH.exists():
    print(f"Checkpoint not found: {CHECKPOINT_PATH}")
    print("Skip this section. Train first or set CHECKPOINT_PATH to an existing file.")
else:
    # Pick a sample (prefer test if available)
    sample = test_pairs[0] if test_pairs else train_pairs[0]
    DATA_TEST = DataConfig(
        ie2d_res_path=sample.ie2d_res,
        target_matrix_path=sample.target_matrix,
        value_kind="auto",
        current_a=1.0,
    )
    raw_t = open_training_data(DATA_TEST)
    prep_t = preprocess_data(raw_t, GRID)

    model_t = init_model(prep_t, MODEL_CFG, checkpoint_path=CHECKPOINT_PATH, strict=True)
    pred = predict_mask(model_t, prep_t, device=TRAIN_CFG.device)
    target = prep_t.sample.target_matrix_raw

    # Visualization
    fig = plot_target_vs_prediction(
        target,
        pred,
        title=f"Trained inference: {sample.target_matrix.stem}",
        x_coords=prep_t.sample.x_coords,
        z_coords=prep_t.sample.z_coords,
    )
    plt.show()

    # Metrics
    err = pred - target
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    den = np.maximum(np.abs(target), 1e-6)
    mape = float(np.mean(np.abs(err) / den) * 100.0)
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((target - np.mean(target)) ** 2))
    r2 = 1.0 - ss_res / (ss_tot + 1e-12)

    print(f"MAE : {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAPE: {mape:.3f}%")
    print(f"R2  : {r2:.6f}")

## TensorBoard

```bash
tensorboard --logdir iternet/runs
```

Валидационные картинки сохраняются рядом с логами в:

- `iternet/runs/<timestamp>/val_images/epoch_0000/`, `epoch_0001/`, ...